# 10 · Test what the process can access

**Skill: what the agent should do. Policy: what its process may access.**

We prepare a policy example, then probe the existing OpenClaw sandbox. The newly written policy is not deployed by this lesson.

**Who does what:** Nano coordinates the policy tools. OpenShell enforces the existing sandbox boundaries. Python makes real HTTP and file-access probes; this lesson needs no Lightning generation.

###  sandbox policy and enforcement evidence

[OpenShell architecture](https://docs.nvidia.com/openshell/about/architecture). The workshop was tested with OpenShell **0.0.85** and `best_effort` Landlock compatibility; current upstream documentation targets a newer release. Use the [terminal setup guide](../run/claw/terminal/README.md) for this pinned example.

finish 08/09, start the existing `nato-openclaw-demo` sandbox, and verify the model endpoint. Then prepare the student policy and run the live probes. These operations require host Docker/OpenShell access; this lesson does not generate model answers.

**What is actually tested:** [chapter_worker.policy_probes](agent/chapter_worker.py) checks `/health` on the host, then checks allowed `/v1/models`, denied `/health`, a canary write denial, and allowed scratch writes from the sandbox. It first verifies that ordinary Unix permissions allow the canary write. A timeout does not satisfy a policy-denial check.

the generated student policy/image context is not deployed. The probes test the existing sandbox policy. Changing a YAML file without applying it cannot change enforcement. The [host-worker diagram](README.md#how-the-notebook-helpers-execute) also identifies which lesson computations occur outside the sandbox.

For sandbox setup and an operator-led policy change, work through [OpenClaw inside OpenShell](../run/claw/terminal/README.md#change-and-test-access). Run the notebook probes again after restoring the original policy.


### Start here
Run the short cells in order. Helpers execute the existing lesson code and keep detailed logs expandable. All displayed results come from your run. Starting again creates a fresh output folder.

In [ ]:
from pathlib import Path
import sys, json
from IPython.display import JSON, display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "course/common.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from course.agent.lesson_walkthrough import LessonWalkthrough, table, bars, loss_chart, show_skill_library
lesson = LessonWalkthrough('10-openshell-policy')

### 1 · Inspect the proposed access rules

| Resource | Intended access |
|---|---|
| GET /v1/models | Allow |
| POST /v1/chat/completions | Allow |
| Source files | Read only |
| /tmp | Read and write |

The tool writes a student policy and packages the loop from 09. This prepares files only; it does not build or deploy a new sandbox.

In [ ]:
lesson.step("prepare")
print((lesson.work / "student-policy.yaml").read_text())

### 2 · Probe the existing sandbox

Run four actual checks. A timeout is not counted as a denial. The source-write test uses a dedicated canary whose normal Unix permissions allow writes, then checks whether sandbox policy blocks it. The canary is removed afterward.

The workshop uses OpenShell’s `best_effort` compatibility mode. These results establish the tested boundaries, not universal isolation.

In [ ]:
lesson.step("probe")
checks = lesson.ns["policy_results"]
table(["Operation", "Observed result"], [("List models", checks["allowed_models"]), ("GET /health", checks["sandbox_health_status"]), ("Source write denied", checks["readonly_write_denied"]), ("Scratch write allowed", checks["scratch_write_allowed"])])
display(JSON(checks, expanded=True))

### Check and save this manual run
Validate the artifacts and record this run for the next manual walkthrough. A successful check describes execution and evidence; read the actual quality scores separately.

In [ ]:
manual_result = lesson.finish()
display(JSON(manual_result, expanded=False))

### Grow the skill library
The library now includes this lesson’s skill, **workshop-policy-check**. Open the actual instructions below.

**`inspect-policy` → `probe-policy` → `check-policy`**

In [ ]:
show_skill_library('10-openshell-policy')

### Prepare OpenClaw
Run once for each new demo. This cell installs the growing skill library in a fresh sandbox workspace and starts a background host worker for this lesson’s three actions. **It does not ask the agent to do anything yet.**

Here, **installing a skill means placing its `SKILL.md` instruction file in OpenClaw’s workspace**. The current lesson’s source is `course/agent/skills/workshop-policy-check/SKILL.md` in this repository.

Each run gets this structure inside the OpenShell sandbox (launcher files omitted):

```text
/home/sandbox/course-runs/openclaw-<run-id>/
├── openclaw.json                 # points OpenClaw at this run’s workspace
├── chapter_tool.py              # helper invoked by the agent
├── chapter.json                 # this lesson’s task and allowed actions
└── workspace/
    ├── AGENTS.md                 # workspace instructions
    └── skills/
        ├── workshop-policy-check/
        │   └── SKILL.md
        └── ...                  # skills from earlier notebooks
```

OpenClaw discovers skills under `workspace/skills/`. The task you paste in the next step tells the agent which skill to read and supplies the full path to `chapter_tool.py` (the skill’s `<tool_path>` placeholder).

The agent invokes the helper through its execution tool. The helper forwards the selected action to the host worker, which runs the fixed lesson code and returns the result. Nano follows the skill and coordinates the calls.

Complete these earlier **OpenClaw demos** first: `08-agent-skills`, `09-agent-loop`. This run copies their latest verified agent artifacts; its inspect action reports the inputs.

Start the TUI task within **15 minutes** and keep this run’s `terminal_run` value for collecting results. If the worker expires, prepare a new run. The local run directory stores host results; the installed skills live at the sandbox paths printed below.


In [ ]:
from course.agent.chapter_terminal import prepare_chapter_terminal, collect_chapter_terminal
from course.agent.notebook_terminal import open_terminal
terminal_run = prepare_chapter_terminal('10-openshell-policy')
print("Local run directory:", terminal_run["directory"])
print("OpenClaw sandbox workspace:", terminal_run["remote"] + "/workspace")
print("Installed skill:", terminal_run["remote"] + "/workspace/skills/workshop-policy-check/SKILL.md")
print("Python helper:", terminal_run["remote"] + "/chapter_tool.py")
print("\nThe agent will run these commands individually:")
for action in ['inspect-policy', 'probe-policy', 'check-policy']:
    print("python3 " + terminal_run["remote"] + "/chapter_tool.py " + action)

### Open the TUI and paste the task
Exit the previous TUI with `/exit` first. This cell opens a terminal on the workshop computer’s desktop. Copy only the generated task into OpenClaw.

The terminal label is **Nano**, its lightweight orchestration model. Nano coordinates the policy tools. OpenShell enforces the existing sandbox boundaries. Python makes real HTTP and file-access probes; this lesson needs no Lightning generation.

In [ ]:
open_terminal(terminal_run)
print("\nPASTE THIS INTO OPENCLAW:\n")
print(terminal_run["prompt"])

### Bring the evidence back
After the TUI finishes, run this cell. It retrieves that session’s trace and checks the artifacts and skill reads. If Run All reaches this cell before you finish the TUI task, run it again afterward.

**Look for:** List models: allowed; /health: denied; source write: denied; scratch write: allowed.

In [ ]:
agent_result = collect_chapter_terminal(terminal_run)
display(JSON(agent_result, expanded=True))

**What you demonstrated:** A skill instruction is not an access control. Check actual allowed and denied operations.

Exit the TUI with `/exit` when done.